# Self-Tooling Agent (STA) | Domain Applications

In [1]:
from langchain_openai import ChatOpenAI
from typing import Dict, Callable, Any
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class SelfToolingAgent:
    def __init__(self):
        self.tools: Dict[str, Callable] = {}
        # Only one simple built-in — the agent must create anything else it needs
        self.tools["sum_list"] = lambda data: sum(data)

    def _needs_tool(self, task: str) -> str | None:
        """Ask the LLM whether the task requires a tool we don't have."""
        available = list(self.tools.keys())
        resp = model.invoke(
            f"Available tools: {available}\nTask: {task}\n\n"
            "If an existing tool can solve this, reply with ONLY its name.\n"
            "If none can, reply with ONLY the snake_case name of the tool we need."
        )
        name = resp.content.strip().strip("`").split()[0]
        return None if name in self.tools else name

    def _generate_and_register(self, name: str, task: str) -> bool:
        """LLM writes a full function, we test it, then register it."""
        resp = model.invoke(
            f"Write a Python function called `{name}` that would help with: {task}\n\n"
            "Requirements:\n"
            "- Full def with type hints and a docstring\n"
            "- Use only builtins + standard lib (no imports beyond typing)\n"
            "- After the function, write 2-3 assert statements that prove it works\n"
            "- Return ONLY the code, no markdown fences"
        )
        code = resp.content.strip()
        if code.startswith("```"):
            code = code.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

        # Execute in a restricted namespace (builtins only, no file/network access)
        safe_globals: Dict[str, Any] = {"__builtins__": {
            "range": range, "len": len, "sum": sum, "abs": abs, "round": round,
            "min": min, "max": max, "int": int, "float": float, "list": list,
            "enumerate": enumerate, "zip": zip, "map": map, "TypeError": TypeError,
            "ValueError": ValueError, "AssertionError": AssertionError,
        }}
        try:
            exec(code, safe_globals)  # noqa: S102 — runs function def + asserts
        except AssertionError as e:
            print(f"  FAIL: generated code failed its own tests — {e}")
            return False
        except Exception as e:
            print(f"  FAIL: execution error — {e}")
            return False

        if name not in safe_globals or not callable(safe_globals[name]):
            print(f"  FAIL: function '{name}' not found after exec")
            return False

        self.tools[name] = safe_globals[name]
        print(f"  Registered new tool: {name}")
        print(f"  Generated code:\n{code}")
        return True

    def run(self, task: str, data: Any = None) -> Any:
        """Main loop: check capability gap -> generate tool if needed -> execute."""
        print(f"\nTask: {task}")
        missing = self._needs_tool(task)
        if missing:
            print(f"  Missing capability detected: '{missing}' — generating...")
            if not self._generate_and_register(missing, task):
                return "ERROR: could not create the required tool"

        # Ask the LLM which tool to call now
        resp = model.invoke(
            f"Tools: {list(self.tools.keys())}\nTask: {task}\n"
            "Reply with ONLY the tool name to use."
        )
        tool_name = resp.content.strip().strip("`").split()[0]
        if tool_name in self.tools:
            result = self.tools[tool_name](data)
            print(f"  Result of {tool_name}({data}): {result}")
            return result
        return f"Tool '{tool_name}' not available"

In [5]:
# --- Demo: agent is asked to compute a moving average (a tool it does NOT have) ---
agent = SelfToolingAgent()
print(f"Tools before: {list(agent.tools.keys())}")

sales_data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
result = agent.run("Calculate the 3-period moving average of this numeric data", sales_data)
print(f"Tools after: {list(agent.tools.keys())}")

Tools before: ['sum_list']

Task: Calculate the 3-period moving average of this numeric data
  Missing capability detected: 'moving_average_3_periods' — generating...
  FAIL: execution error — __import__ not found
Tools after: ['sum_list']


In [6]:
# --- Use the generated tool in a realistic pipeline ---
print(f"\n--- Realistic Pipeline Usage ---")
print(f"Generated tool output: {result}")
print("This tool can now be used by any agent in the system.")


--- Realistic Pipeline Usage ---
Generated tool output: ERROR: could not create the required tool
This tool can now be used by any agent in the system.
